### Retrieve an ET time series from a multi-polygon
---


This notebook guides you through the OpenET multi-polygon API endpoint in order to extract monthly time series for 4 eddy covariance tower locations.

The eddy covariance sites include: US-Twt (rice), US-Tw3 (alfalfa), US-Bi1 (alfalfa), and US-Bi2 (corn). More information on each site and the ET data used in this example can be found in the OpenET benchmark ET dataset ([Volk et al., 2023](https://www.data-in-brief.com/article/S2352-3409(23)00393-1/fulltext)). The entire benchmark ET dataset, including daily and monthly datasets, can be accessed at this [link](https://zenodo.org/records/7636781).



### Get Set Up ###
* Import necessary Python libraries
* Load your API key.



In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import getpass
import glob

* Find your API Key from the [OpenET account](https://auth.etdata.org/).

* Copy your API Key to paste below.

In [ ]:
YOUR_API_KEY = input("Paste your API key here: ")
# set your API key before making the request
header = {"Authorization": YOUR_API_KEY}

### Construct argument to pass to the API.

* To get data from the OpenET API, certain parameters are required including the **date_range**, **interval**, **geometry**, **model**, **variable**, **units** and **reference_et**.

* Fore the multi-polygon end point, you must load geometry to Google Earth Engine (GEE). This example uses the footprints from each of the 4 eddy covariance towers that was created for the training:
projects/openet-training-assets/assets/cadwr/ca_delta_footprints

* Example 2 will walk through step-by-step how to upload geometry to GEE

* For more information on appropriate parameters to pass to the API end point: https://openet.gitbook.io/docs/reference/api-reference#accepted-parameters



In [ ]:
# endpoint arguments
args = {
  "date_range": [
    "2013-01-01",          # start date
    "2019-12-31"           # end date
  ],
  "interval": "monthly",   # frequency of data: daily or monthly
                           # * daily data are capped at 61 days *
  "asset_id": "projects/openet-training-assets/assets/cadwr/ca_delta_footprints", # < google earth engine asset
  "attributes": [
    "site"
  ],                       # Must include a unique identifier for each polygon
  "reducer": "mean",       # Pixel aggregation method within each polygon
  "model": "ensemble",     # Model options: Ensemble, geeSEBAL, SSEBop, SIMS, DisALEXI, PTJPL, or eeMETRIC.
  "variable": "ET",        # Variable options: ET, EToF, ETo, PR, NDVI
  "reference_et": "cimis", # ETo options: CIMiS, gridMET
  "units": "mm",           # Unit options: mm, in
  "version":2.1
}

# query the api
resp = requests.post(
    headers=header,
    json=args,
    url="https://california.openet-api.org/raster/timeseries/multipolygon"
)

# store the result
output = resp.json()

### Inspect the output

* The output includes a 'url' which returns a csv file.
* Run the cell below:
* You should see a hyperlink printed: If you click the hyperlink a table will be downloaded to your computer.

In [ ]:
output

#### Inspect OpenET table

* Read the url as a dataframe and inspect the table


In [ ]:
# covert to a dataframe
df = pd.read_csv(output["url"], parse_dates=["time"])
print(df.head(4))

df.to_csv("ca_delta_ensemble_et_values.csv") #< change the file name + path here

For the table above:

* Each day includes 4 rows, one for each site.

* The et column is in units of mm, defined in the API argument. The hectares column indicates the size of the polygon footprint at each site. Megalites is the volume associated with this area.

In [ ]:
# ensure "time" column is datetime type
df["time"] = pd.to_datetime(df["time"])

# plotting style + figures ize
plt.style.use('ggplot')

#assigning colors to sites for interpretation
colors =['blue','orange','green','purple']

i=0
plt.figure(figsize=(12, 6))

for id_value in df["site"].unique():

  # filter the DataFrame for the current "id"
  df_filtered = df[df["site"] == id_value]
  # Constraint plot to only period where both data are present

  # plot the ET values for this "id"
  plt.plot(df_filtered["time"], df_filtered["et"], label=f"Site: {id_value}",linestyle='-',lw=2,c=colors[i])

  i+=1

plt.title(f"ET Values by Site")
plt.ylabel("ET (mm)")
plt.grid(True, linestyle="--", linewidth=0.5, color="grey")
plt.legend(ncol=4)
plt.show()



---



# Combine with EC data from each tower.
*This step can also be completed outside of Python.*

### Required Step: Upload EC data to the contents.
* Make sure you downloaded the Example 1 folder to your compture. You will need to upload files from the `eddy_covariance_data/` folder

* On the left side of Collab there is a files icon. Click the folder, then select the 'Upload to session storage' button on the top. The icon looks like a a piece of paper with an up arrow. You should be prompted to upload a file.

* Upload the following csv files: US-Bi1_monthly_data.csv, US-Bi2_monthly_data.csv, US-Tw3_monthly_data.csv, & US-Twt_monthly_dat to the current session before proceeding.

* Google Collab is great, but you are unable to save files session to session. If you are running this from a local computer, you can altenratively point to the `eddy_covariance_data/` folder to read the files below.


### Run the code below to generate time series plots for each site

* The EC data are read in and the dates are harmonized to the first of the month.
* The OpenET data are filtered for the period of record that overlaps with when the high quality EC data are available.
* A time series is then plotted for each site.

*The code below includes a check to make sure EC file exists.*

In [ ]:
# ensure "time" column is datetime type
df["time"] = pd.to_datetime(df["time"])

i=0
for id_value in df["site"].unique():
  ec_file = f"eddy_covariance_data/{id_value}_monthly_data.csv"
  if Path(ec_file).is_file():
    plt.figure(figsize=(6, 3))

    ec_df = pd.read_csv(ec_file)
    ec_df["date"] = pd.to_datetime(ec_df["date"])

    #This next line aligns the date columns
    #Each month is represented as 1st day of eachmonth
    ec_df['date'] = ec_df['date'] - pd.offsets.MonthEnd() + pd.offsets.MonthBegin()

    ec_df = ec_df[(ec_df.date>'2013-01-01')&(ec_df.date<'2020-01-01')]

    # filter the DataFrame for the current "id"
    df_filtered = df[df["site"] == id_value]
    # Constraint plot to only period where both data are present
    df_filtered = df_filtered.loc[(df_filtered['time'] > ec_df.date.values[0]) & (df_filtered['time'] < ec_df.date.values[-1])]

    # plot the ET values for this "id"
    plt.plot(df_filtered["time"], df_filtered["et"], label=f"Site: {id_value}",linestyle='-',lw=2,c=colors[i])
    plt.plot(ec_df["date"], ec_df["ET_corr"], label=f"EC: {id_value}",linestyle=':',c='k',lw=2)
    plt.title(f"OpenET vs {id_value} EC measurements")
    plt.ylabel("ET (mm)")
    plt.grid(True, linestyle="--", linewidth=0.5, color="grey")
    plt.legend(ncol=2,loc='upper right')
    # plt.savefig(f'OpenET_vs_EC_comp_+{id_value}.png') # un comment line if you want to save a figure
    plt.show()

  else:
    print(f"File {ec_file} does not exist. \n\nUpload the {ec_file.split('/')[-1]} to contents following the instructions above\n\n")
  i+=1

### Run the code below to generate a multi-site scatter plot

* The block of code below reads in the EC data and combines with the OpenET data.
* For each tower monthly pairs are plotted against the 1 to 1 line.

In [ ]:
# ensure "time" column is datetime type
df["time"] = pd.to_datetime(df["time"])

# plotting
plt.figure(figsize=(6, 6))
ones = np.arange(0,250,5)
# loop through each unique "id"
i=0
for id_value in df["site"].unique():
  ec_file = f"eddy_covariance_data/{id_value}_monthly_data.csv"
  ec_df = pd.read_csv(ec_file)
  ec_df["date"] = pd.to_datetime(ec_df["date"])
  ec_df['date'] = ec_df['date'] - pd.offsets.MonthEnd() + pd.offsets.MonthBegin()
  ec_df.set_index("date", inplace=True)

  # filter the DataFrame for the current "id"
  df_filtered = df[df["site"] == id_value]
  df_filtered.set_index("time", inplace=True)
  merged_df = pd.merge( ec_df, df_filtered, left_index=True, right_index=True, how='inner')
  # plot the ET values for this "id"
  plt.plot(ones, ones, 'k--')
  plt.scatter(merged_df["ET_corr"], merged_df["et"], s=8,label=f"Site: {id_value}",c=colors[i])
  i+=1


plt.title("ET Values Over Time by Site")
plt.xlabel("ET$_{obs}$ (mm)")
plt.ylabel("ET$_{OPENET}$ (mm)")
plt.grid(True, linestyle="--", linewidth=0.5, color="grey")
plt.legend()
plt.tight_layout()
plt.show()
